# DE pseudobulk

Run donor-level pseudobulk differential expression for broad classes, selected glia subtypes, and selected excitatory subtypes. The contrast label follows the saved file name, while the log2 fold-change is interpreted as the second condition relative to the first condition.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PSEUDOBULK_DIR = PROJECT_ROOT / 'results' / '10_Pseudobulk'
DE_DIR = PROJECT_ROOT / 'results' / '13_DE_pseudobulk'

INPUT_DIRS = {
    'broad': PSEUDOBULK_DIR / 'broad',
    'glia_subtypes': PSEUDOBULK_DIR / 'glia_subtypes',
    'excitatory_subtypes': PSEUDOBULK_DIR / 'excitatory_subtypes',
}
OUTPUT_DIRS = {
    'broad': DE_DIR / 'broad',
    'glia_subtypes': DE_DIR / 'glia_subtypes',
    'excitatory_subtypes': DE_DIR / 'excitatory_subtypes',
}
for out_dir in OUTPUT_DIRS.values():
    out_dir.mkdir(parents=True, exist_ok=True)


## Settings


In [3]:
RUN_DE = True
OVERWRITE_EXISTING_RESULTS = True
MIN_DONORS_PER_CONDITION = 2
MIN_CELLS_PER_DONOR_GROUP = 20

GROUPS = {
    'broad': ['Excitatory', 'Inhibitory', 'Glia', 'Vascular'],
    'glia_subtypes': ['Glia_Oligo', 'Glia_OPC', 'Glia_Micro', 'Glia_Astro'],
    'excitatory_subtypes': [
        'Ex_L5_PCP4_NXPH2',
        'Ex_L5_VAT1L_THSD4',
        'Ex_L5_L6_THEMIS_NR4A2',
        'Ex_L5_L6_THEMIS_TMEM233',
        'Ex_L6_TLE4_CCBE1',
        'Ex_L6_TLE4_MEGF11',
    ],
}

# Files are labelled first_vs_second; PyDESeq2 contrast is second relative to first.
COMPARISONS = [
    ('Control', 'sALS'),
    ('Control', 'c9ALS'),
    ('sALS', 'c9ALS'),
]


## Load pseudobulk and inspect donor coverage


In [4]:
def load_pseudobulk(group_name, input_dir):
    counts = pd.read_csv(input_dir / f'{group_name}_counts.csv', index_col=0)
    meta = pd.read_csv(input_dir / f'{group_name}_meta.csv', index_col=0)
    counts.index = counts.index.astype(str)
    meta.index = meta.index.astype(str)
    common = counts.index.intersection(meta.index)
    counts = counts.loc[common]
    meta = meta.loc[common].copy()
    return counts, meta

coverage_rows = []
available_groups = {level: [] for level in GROUPS}

for level, group_names in GROUPS.items():
    for group_name in group_names:
        counts_path = INPUT_DIRS[level] / f'{group_name}_counts.csv'
        meta_path = INPUT_DIRS[level] / f'{group_name}_meta.csv'
        if not counts_path.exists() or not meta_path.exists():
            coverage_rows.append({'level': level, 'group': group_name, 'condition': pd.NA, 'n_donors': 0, 'available': False})
            continue

        counts, meta = load_pseudobulk(group_name, INPUT_DIRS[level])
        if 'n_cells' in meta.columns:
            meta = meta[meta['n_cells'] >= MIN_CELLS_PER_DONOR_GROUP]
            counts = counts.loc[meta.index]
        available_groups[level].append(group_name)
        for condition, n_donors in meta['condition'].astype(str).value_counts().sort_index().items():
            coverage_rows.append({'level': level, 'group': group_name, 'condition': condition, 'n_donors': int(n_donors), 'available': True})

coverage = pd.DataFrame(coverage_rows)
display(coverage.pivot_table(index=['level', 'group'], columns='condition', values='n_donors', fill_value=0, aggfunc='sum'))


condition                                    Control  c9ALS  sALS
level               group                                        
broad               Excitatory                     4      4     4
                    Glia                           4      4     4
                    Inhibitory                     4      4     4
                    Vascular                       3      4     3
excitatory_subtypes Ex_L5_L6_THEMIS_NR4A2          3      4     2
                    Ex_L5_L6_THEMIS_TMEM233        4      4     4
                    Ex_L5_PCP4_NXPH2               3      4     2
                    Ex_L5_VAT1L_THSD4              1      2     1
                    Ex_L6_TLE4_MEGF11              4      4     4
glia_subtypes       Glia_Astro                     4      4     4
                    Glia_Micro                     4      4     4
                    Glia_OPC                       4      4     4
                    Glia_Oligo                     4      4     4

## Run DESeq2 contrasts


In [5]:
def run_deseq2(counts, meta, numerator, denominator):
    keep = meta['condition'].astype(str).isin([denominator, numerator])
    counts_use = counts.loc[keep].round().astype(int)
    meta_use = meta.loc[keep, ['condition']].copy()
    meta_use['condition'] = pd.Categorical(meta_use['condition'].astype(str), categories=[denominator, numerator])

    donor_counts = meta_use['condition'].value_counts()
    if (donor_counts < MIN_DONORS_PER_CONDITION).any():
        return None, f'too_few_donors: {donor_counts.to_dict()}'

    expressed = counts_use.sum(axis=0) > 0
    counts_use = counts_use.loc[:, expressed]
    if counts_use.shape[1] == 0:
        return None, 'no_expressed_genes'

    dds = DeseqDataSet(counts=counts_use, metadata=meta_use, design='~condition', refit_cooks=True)
    dds.deseq2()
    stats = DeseqStats(dds, contrast=['condition', numerator, denominator])
    stats.summary()
    result = stats.results_df.reset_index()
    result = result.rename(columns={result.columns[0]: 'gene'})
    return result, None

skipped = []
written = []

if RUN_DE:
    for level, group_names in available_groups.items():
        for group_name in group_names:
            counts, meta = load_pseudobulk(group_name, INPUT_DIRS[level])
            if 'n_cells' in meta.columns:
                meta = meta[meta['n_cells'] >= MIN_CELLS_PER_DONOR_GROUP]
                counts = counts.loc[meta.index]

            for denominator, numerator in COMPARISONS:
                comparison = f'{denominator}_vs_{numerator}'
                out_path = OUTPUT_DIRS[level] / f'{group_name}_{comparison}.csv'
                if out_path.exists() and not OVERWRITE_EXISTING_RESULTS:
                    written.append({'level': level, 'group': group_name, 'comparison': comparison, 'path': str(out_path), 'status': 'already_exists'})
                    continue

                result, reason = run_deseq2(counts, meta, numerator=numerator, denominator=denominator)
                if reason:
                    skipped.append({'level': level, 'group': group_name, 'comparison': comparison, 'reason': reason})
                    continue

                result.to_csv(out_path, index=False)
                written.append({'level': level, 'group': group_name, 'comparison': comparison, 'path': str(out_path), 'status': 'written'})

written = pd.DataFrame(written)
skipped = pd.DataFrame(skipped)
display(written)
if not skipped.empty:
    display(skipped)


Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 2.16 seconds.

Fitting dispersion trend curve...
... done in 0.42 seconds.

Fitting MAP dispersions...
... done in 2.57 seconds.

Fitting LFCs...
... done in 2.22 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.90 seconds.



Log2 fold change & Wald test p-value: condition sALS vs Control
                   baseMean  log2FoldChange     lfcSE      stat    pvalue  \
TNFRSF4            6.828276       -1.585346  0.613595 -2.583700  0.009775   
TNFRSF18          26.313131       -1.203761  0.480726 -2.504050  0.012278   
ATAD3B           282.673015       -0.490314  0.208274 -2.354180  0.018564   
ENSG00000260179    1.397099       -0.715433  1.328995 -0.538327  0.590351   
ENSG00000234396   23.074303       -0.506082  0.371593 -1.361925  0.173222   
...                     ...             ...       ...       ...       ...   
ENSG00000271254   66.519065        0.521118  0.276164  1.886987  0.059162   
U1-4               0.123783        1.447359  3.843115  0.376611  0.706463   
ENSG00000268674    0.075415       -0.392461  3.893677 -0.100795  0.919714   
ENSG00000277475    0.106124       -0.392461  3.893677 -0.100795  0.919714   
U1-5               1.207666        0.567894  1.463895  0.387933  0.698065   

           

Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


...

Log2 fold change & Wald test p-value: condition c9ALS vs Control
                  baseMean  log2FoldChange     lfcSE      stat    pvalue  \
TNFRSF4           0.771891        0.383658  1.864520  0.205768  0.836972   
TNFRSF18          1.524558       -1.585827  1.334231 -1.188570  0.234609   
ATAD3B           39.594843       -0.315772  0.299120 -1.055669  0.291120   
ENSG00000260179   0.263786       -1.579953  3.774022 -0.418639  0.675480   
ENSG00000234396   4.107049       -0.539871  0.763087 -0.707483  0.479267   
...                    ...             ...       ...       ...       ...   
ENSG00000277196   0.065448       -0.639199  3.885755 -0.164498  0.869339   
ENSG00000278817   8.673988       -1.054283  0.632752 -1.666187  0.095676   
ENSG00000278384   3.596068       -0.585103  0.900315 -0.649887  0.515765   
ENSG00000276345   0.207885        0.114809  3.734100  0.030746  0.975472   
ENSG00000271254   9.454864       -0.869885  0.607600 -1.431673  0.152237   

                     p

Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 1.80 seconds.

Fitting dispersion trend curve...
... done in 0.36 seconds.

Fitting MAP dispersions...
... done in 1.87 seconds.

Fitting LFCs...
... done in 1.74 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.80 seconds.



Log2 fold change & Wald test p-value: condition c9ALS vs sALS
                  baseMean  log2FoldChange     lfcSE      stat    pvalue  \
TNFRSF4           0.474690        0.950632  2.494495  0.381092  0.703135   
TNFRSF18          0.920850       -0.988971  1.640790 -0.602741  0.546681   
ATAD3B           24.664870        0.361352  0.324279  1.114326  0.265139   
ENSG00000234396   2.538707        0.134098  0.920874  0.145621  0.884221   
MTND1P23          0.770629        0.115185  2.253638  0.051111  0.959237   
...                    ...             ...       ...       ...       ...   
ENSG00000277196   0.125740       -1.379017  3.878162 -0.355585  0.722151   
ENSG00000278817   4.238781        0.117831  0.719841  0.163690  0.869975   
ENSG00000278384   3.155379       -0.952522  1.002026 -0.950596  0.341810   
ENSG00000276345   0.241930       -0.625389  3.546885 -0.176321  0.860042   
ENSG00000271254   8.845535       -1.239213  0.619283 -2.001046  0.045387   

                     padj

,level,group,comparison,path,status
0,broad,Excitatory,Control_vs_sALS,/Users/wangj/Documents/Computational_Biology/P...,written
1,broad,Excitatory,Control_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...,written
2,broad,Excitatory,sALS_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...,written
3,broad,Inhibitory,Control_vs_sALS,/Users/wangj/Documents/Computational_Biology/P...,written
4,broad,Inhibitory,Control_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...,written
5,broad,Inhibitory,sALS_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...,written
6,broad,Glia,Control_vs_sALS,/Users/wangj/Documents/Computational_Biology/P...,written
7,broad,Glia,Control_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...,written
8,broad,Glia,sALS_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...,written
9,broad,Vascular,Control_vs_sALS,/Users/wangj/Documents/Computational_Biology/P...,written


,level,group,comparison,reason
0,excitatory_subtypes,Ex_L5_VAT1L_THSD4,Control_vs_sALS,"too_few_donors: {'Control': 1, 'sALS': 1}"
1,excitatory_subtypes,Ex_L5_VAT1L_THSD4,Control_vs_c9ALS,"too_few_donors: {'c9ALS': 2, 'Control': 1}"
2,excitatory_subtypes,Ex_L5_VAT1L_THSD4,sALS_vs_c9ALS,"too_few_donors: {'c9ALS': 2, 'sALS': 1}"


## Output inventory


In [11]:
inventory = []
for level, out_dir in OUTPUT_DIRS.items():
    for path in sorted(out_dir.glob('*.csv')):
        parts = path.stem.rsplit('_', 3)
        comparison = '_'.join(parts[-3:]) if len(parts) >= 4 else pd.NA
        group_name = path.stem[:-(len(comparison) + 1)] if pd.notna(comparison) else path.stem
        inventory.append({'level': level, 'group': group_name, 'comparison': comparison, 'path': str(path)})

inventory = pd.DataFrame(inventory)
inventory.to_csv(DE_DIR / 'de_output_inventory.csv', index=False)
display(inventory)


,level,group,comparison,path
0,broad,Excitatory,Control_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...
1,broad,Excitatory,Control_vs_sALS,/Users/wangj/Documents/Computational_Biology/P...
2,broad,Excitatory,sALS_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...
3,broad,Glia,Control_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...
4,broad,Glia,Control_vs_sALS,/Users/wangj/Documents/Computational_Biology/P...
5,broad,Glia,sALS_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...
6,broad,Inhibitory,Control_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...
7,broad,Inhibitory,Control_vs_sALS,/Users/wangj/Documents/Computational_Biology/P...
8,broad,Inhibitory,sALS_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...
9,broad,Vascular,Control_vs_c9ALS,/Users/wangj/Documents/Computational_Biology/P...


## Interpretation note

For a file named `Control_vs_c9ALS`, the contrast is `c9ALS` relative to `Control`. A positive log2FoldChange means higher expression in c9ALS than Control; a negative value means lower expression in c9ALS than Control.
